# EduTrack Analytics - Analyse Exploratoire des Donnees (EDA)

Ce notebook explore les donnees academiques synthetiques: nettoyage, statistiques
descriptives, distributions, correlations et segmentation des etudiants par niveau.

Les fichiers sont generes par `data/generate_synthetic.py` et lus depuis `../data/samples/`.
Les colonnes sont en francais.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
SAMPLES = '../data/samples'

## 1. Chargement des donnees

In [ ]:
etudiants = pd.read_csv(f'{SAMPLES}/etudiants.csv')
notes = pd.read_csv(f'{SAMPLES}/notes.csv')
absences = pd.read_excel(f'{SAMPLES}/absences.xlsx')
print('etudiants:', etudiants.shape, '| notes:', notes.shape, '| absences:', absences.shape)
etudiants.head()

## 2. Nettoyage
Suppression des doublons et controle des valeurs (notes 0..20, dates, champs manquants).

In [ ]:
# Doublons
print('Doublons notes:', notes.duplicated().sum())
notes = notes.drop_duplicates()

# Notes hors intervalle 0..20
hors_bornes = ~notes['note'].between(0, 20)
print('Notes hors 0..20:', hors_bornes.sum())
notes = notes[~hors_bornes]

# Dates d'absence invalides
absences['date'] = pd.to_datetime(absences['date'], errors='coerce')
print('Dates invalides:', absences['date'].isna().sum())

# Valeurs manquantes etudiants
print('Noms manquants:', etudiants['nom'].isna().sum())
etudiants = etudiants.dropna(subset=['nom'])
etudiants.shape, notes.shape, absences.shape

## 3. Statistiques descriptives par module

In [ ]:
par_module = notes.groupby('nom_module')['note'].agg(
    moyenne='mean', mediane='median', min='min', max='max', variance='var', ecart_type='std'
).round(2).sort_values('moyenne')
par_module['taux_reussite'] = (
    notes.assign(ok=notes['note'] >= 10).groupby('nom_module')['ok'].mean().round(3)
)
par_module

## 4. Distribution des notes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(notes['note'], bins=range(0, 21), color='#1d4ed8', edgecolor='white')
ax.axvline(10, color='red', linestyle='--', label='Seuil de validation')
ax.set_xlabel('Note'); ax.set_ylabel('Nombre de notes'); ax.set_title('Distribution des notes')
ax.legend(); plt.show()

## 5. Correlation entre absences et resultats

In [ ]:
moyenne = notes.groupby('code_etudiant')['note'].mean().rename('moyenne')
ecart = notes.groupby('code_etudiant')['note'].std().rename('ecart_notes')
abs_h = absences[absences['type_absence'] == 'absence'].groupby('code_etudiant')['heures'].sum().rename('absences_h')
retards_h = absences[absences['type_absence'] == 'retard'].groupby('code_etudiant')['heures'].sum().rename('retards_h')

feat = pd.concat([moyenne, ecart, abs_h, retards_h], axis=1).fillna(0)
corr = feat.corr()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center')
fig.colorbar(im); ax.set_title('Matrice de correlation'); plt.tight_layout(); plt.show()
print('Correlation absences / moyenne:', round(corr.loc['absences_h', 'moyenne'], 3))

## 6. Segmentation des etudiants par niveau
Classement de chaque etudiant en excellent / stable / moyen / fragile / a risque a partir de sa moyenne.

In [ ]:
def segment(m):
    if m >= 16: return 'excellent'
    if m >= 13: return 'stable'
    if m >= 10: return 'moyen'
    if m >= 8:  return 'fragile'
    return 'a_risque'

feat['segment'] = feat['moyenne'].apply(segment)
counts = feat['segment'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(counts.index, counts.values, color='#1d4ed8')
ax.set_ylabel("Nombre d'etudiants"); ax.set_title('Repartition des etudiants par segment')
plt.show()

## 7. Conclusions

- La distribution des notes est centree autour du seuil de validation, avec une part notable d'etudiants sous la moyenne.
- La correlation negative entre les heures d'absence et la moyenne confirme l'impact de l'assiduite sur la performance.
- La segmentation par niveau met en evidence les groupes d'etudiants a accompagner en priorite.

Ces analyses sont reprises dans l'application (tableau de bord, alertes, segmentation) et dans le rapport genere automatiquement.